# Неделя 5 — Word2Vec, GloVe и fastText

Цель: обучить собственные векторы, найти соседей, выполнить арифметику и сравнить покрытие. GloVe предобучен на внешнем корпусе; равенства объёма обучения здесь нет.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'nlp_project').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown
from nlp_project.core import corpus, clean, tokens, metrics, SEED
df = corpus()
train, val, test = [df[df.split == s].copy() for s in ('train', 'validation', 'test')]


In [2]:
from gensim.models import Word2Vec, FastText
sentences = [tokens(t) for t in train.text]
local = Word2Vec(sentences, vector_size=50, min_count=1, window=5, sg=1, workers=1, epochs=40, seed=SEED)
print('Word2Vec / phone:', local.wv.most_similar('phone', topn=5))
print('good + phone - bad:', local.wv.most_similar(positive=['good','phone'], negative=['bad'], topn=5))
ft = FastText.load(str(ROOT / 'models/fasttext.model'))
print('fastText, OOV-вектор:', ft.wv['batteryzzzz'].shape)
examples = json.loads((ROOT / 'reports/semantic_examples.json').read_text())
display(pd.DataFrame({k: str(v.get('phone')) for k,v in examples.items()}, index=['соседи phone']).T)
display(pd.read_json(ROOT / 'reports/embedding_coverage.json').T)


Word2Vec / phone: [('samsung', 0.9905726909637451), ('r', 0.9905542731285095), ('reccomendation', 0.9903759956359863), ('clearly', 0.9903643131256104), ('operates', 0.990263044834137)]
good + phone - bad: [('price', 0.9785988330841064), ('hands', 0.9771685600280762), ('indoors', 0.976347029209137), ('awesome', 0.9749816656112671), ('new', 0.9748637080192566)]
fastText, OOV-вектор: (50,)


,соседи phone
word2vec,"[['samsung', 0.9905727505683899], ['r', 0.9905..."
fasttext,"[['phones', 0.9991323351860046], ['iphone', 0...."
glove,"[['telephone', 0.9342338442802429], ['phones',..."


,vocab,known
word2vec,1131,1131
fasttext,1131,1131
glove,1131,1099


## Выводы

На 589 train-документах Word2Vec/fastText ограничены объёмом и могут давать неинтуитивных соседей. fastText использует n-граммы символов; GloVe покрывает только известные слова. Арифметика не гарантирует осмысленного ответа.